# LAB 5: Pipeline RAG Completo — Docling + OpenSearch + vLLM
## Aula 5 — Docling e Ingestão Inteligente de Documentos
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**⏱️ Duração estimada:** 60 minutos  
**📚 Pré-requisitos:** Labs 1–4 concluídos  
**🎯 Objetivos:**
- Integrar o pipeline Docling com OpenSearch para busca híbrida
- Implementar chunking com metadados ricos para indexação
- Configurar RAG com LangChain usando o corpus jurídico da Aula 5
- Avaliar qualidade da ingestão com consultas jurídicas reais
- Implementar fallback FAISS quando OpenSearch não disponível

In [ ]:
!pip install docling>=2.0.0 sentence-transformers langchain langchain-openai faiss-cpu --quiet
!pip install opensearch-py --quiet
print("✅ Instalação concluída!")

In [ ]:
import sys, warnings, os, json, re
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from sentence_transformers import SentenceTransformer
import numpy as np
print("✅ Ambiente configurado")

## ⚙️ Etapa 1: Configuração do Pipeline

Configuramos o pipeline com fallback automático:
- **Com OpenSearch disponível:** busca híbrida kNN + BM25
- **Sem OpenSearch (Colab padrão):** FAISS para busca vetorial

**⏱️ Tempo estimado: 5 minutos**

In [ ]:
# ─── Configuração ──────────────────────────────────────────────────────────────
USE_OPENSEARCH = False  # Mude para True se tiver OpenSearch disponivel
OPENSEARCH_HOST = 'localhost'
OPENSEARCH_PORT = 9200
OPENSEARCH_INDEX = 'corpus_juridico_aula5'

USE_VLLM = False  # Mude para True se tiver vLLM disponivel
VLLM_URL = 'http://localhost:8000/v1'

# Testar conectividade OpenSearch
if USE_OPENSEARCH:
    try:
        from opensearchpy import OpenSearch
        client = OpenSearch(hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}])
        info = client.info()
        print(f'✅ OpenSearch conectado: {info["version"]["number"]}')
    except Exception as e:
        print(f'❌ OpenSearch não disponível: {e}')
        print('→ Usando fallback FAISS')
        USE_OPENSEARCH = False
else:
    print('⚠️ Modo FAISS ativo (OpenSearch desabilitado)')

# Testar vLLM
if USE_VLLM:
    try:
        import urllib.request
        urllib.request.urlopen(f'{VLLM_URL}/models', timeout=2)
        print(f'✅ vLLM disponível em {VLLM_URL}')
    except Exception:
        print('⚠️ vLLM não disponível — usando LLM simulado')
        USE_VLLM = False

print(f"\nModo de operação: OpenSearch={USE_OPENSEARCH} | vLLM={USE_VLLM}")

## 📚 Etapa 2: Carregar Corpus e Gerar Chunks

Usamos o corpus e os chunks gerados no Lab 4. Se não estiver disponível,
criamos um corpus mínimo inline.

**⏱️ Tempo estimado: 10 minutos**

In [ ]:
# Carregar corpus do Lab 4 ou criar corpus mínimo
CHUNKS_PATH = Path('/tmp/aula5_chunks.json')

if CHUNKS_PATH.exists():
    with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
        chunks_base = json.load(f)
    print(f"✅ Corpus carregado do Lab 4: {len(chunks_base)} chunks")
else:
    # Corpus mínimo inline (caso Lab 4 não tenha sido executado)
    print('⚠️ Corpus do Lab 4 não encontrado — usando corpus mínimo inline')
    chunks_base = [
        {'texto': 'A prisão preventiva exige fundamentação concreta e individualizada. '
                  'A mera referência à gravidade abstrata do delito não é suficiente. '
                  'Súmula 440/STJ. Art. 312 do CPP.',
         'arquivo': 'hc_stj.pdf', 'chunk_index': 0},
        {'texto': 'O crime de descumprimento de medida protetiva (art. 24-A Lei 11.340/2006) '
                  'é formal. Consuma-se com a mera inobservância da ordem judicial. '
                  'O consentimento da vítima não afasta a tipicidade penal.',
         'arquivo': 'lei_maria_penha.pdf', 'chunk_index': 0},
        {'texto': 'A LGPD (Lei 13.709/2018) classifica imagens de videomonitoramento '
                  'policial como dados pessoais sensíveis (art. 5º, II). '
                  'Prazo de retenção: 30 dias (Decreto SP 68.447/2024).',
         'arquivo': 'lgpd_segpublica.pdf', 'chunk_index': 0},
        {'texto': 'Contratação direta por emergência (art. 75, VIII NLLC): '
                  'requer comprovação documental, limitação ao necessário, '
                  'prazo máximo 1 ano, mapa de preços e justificativa do fornecedor.',
         'arquivo': 'licitacao_emergencia.pdf', 'chunk_index': 0},
        {'texto': 'O STF (RE 1.055.941, Tema 990) autorizou o compartilhamento de '
                  'dados do COAF com órgãos de persecução penal sem autorização '
                  'judicial prévia, desde que com comunicação posterior ao juiz.',
         'arquivo': 'lgpd_segpublica.pdf', 'chunk_index': 1},
    ]
    print(f"✅ Corpus mínimo criado: {len(chunks_base)} chunks")

In [ ]:
print("⏳ Carregando modelo BGE-M3...")
modelo = SentenceTransformer('BAAI/bge-m3')
print("✅ Modelo carregado")

# Gerar embeddings
textos = [c['texto'] for c in chunks_base]
print(f"\n⏳ Gerando embeddings para {len(textos)} chunks...")
embeddings = modelo.encode(textos, normalize_embeddings=True, show_progress_bar=True)

for i, chunk in enumerate(chunks_base):
    chunk['embedding'] = embeddings[i].tolist()

print(f"\n✅ Embeddings prontos: shape={embeddings.shape}")

## 🔍 Etapa 3: Indexar no OpenSearch ou FAISS

O pipeline usa OpenSearch (produção) ou FAISS (desenvolvimento/Colab).

**⏱️ Tempo estimado: 10 minutos**

In [ ]:
import faiss

DIM = embeddings.shape[1]  # 1024 para BGE-M3

if USE_OPENSEARCH:
    # Indexar no OpenSearch
    from opensearchpy import OpenSearch
    client = OpenSearch(hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}])

    # Criar índice híbrido
    mapping = {
        'settings': {'index': {'knn': True}},
        'mappings': {'properties': {
            'texto': {'type': 'text', 'analyzer': 'portuguese'},
            'embedding': {'type': 'knn_vector', 'dimension': DIM,
                'method': {'name': 'hnsw', 'space_type': 'cosinesimil', 'engine': 'faiss'}},
            'arquivo': {'type': 'keyword'},
            'chunk_index': {'type': 'integer'},
        }}
    }
    if client.indices.exists(OPENSEARCH_INDEX):
        client.indices.delete(OPENSEARCH_INDEX)
    client.indices.create(OPENSEARCH_INDEX, body=mapping)

    # Indexar documentos
    for chunk in chunks_base:
        client.index(index=OPENSEARCH_INDEX, body={
            'texto': chunk['texto'],
            'embedding': chunk['embedding'],
            'arquivo': chunk.get('arquivo', ''),
            'chunk_index': chunk.get('chunk_index', 0)
        })
    client.indices.refresh(OPENSEARCH_INDEX)
    print(f"✅ {len(chunks_base)} docs indexados no OpenSearch ({OPENSEARCH_INDEX})")

else:
    # Fallback FAISS
    index_faiss = faiss.IndexFlatIP(DIM)
    index_faiss.add(embeddings.astype('float32'))
    print(f"⚠️ Usando FAISS: {index_faiss.ntotal} vetores indexados (dim={DIM})")

## 🤖 Etapa 4: Configurar LLM e Pipeline RAG LangChain

Conectamos o LLM (vLLM ou simulado) ao retriever para criar o pipeline RAG.

**⏱️ Tempo estimado: 15 minutos**

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

def buscar_chunks(query, top_k=5):
    # Recupera chunks relevantes para a query (FAISS ou OpenSearch)
    query_emb = modelo.encode([query], normalize_embeddings=True).astype('float32')

    if USE_OPENSEARCH:
        resposta = client.search(
            index=OPENSEARCH_INDEX,
            body={'query': {'knn': {'embedding': {'vector': query_emb[0].tolist(), 'k': top_k}}}}
        )
        return [hit['_source']['texto'] for hit in resposta['hits']['hits']]
    else:
        D, I = index_faiss.search(query_emb, top_k)
        return [chunks_base[i]['texto'] for i in I[0] if i < len(chunks_base)]

# Prompt RAG jurídico
PROMPT_RAG = ChatPromptTemplate.from_template(
    'Você é um assistente jurídico especializado em direito brasileiro e segurança pública.\n'
    'Responda a pergunta SOMENTE com base nos documentos fornecidos.\n'
    'Se a informação não estiver nos documentos, diga isso claramente.\n\n'
    'DOCUMENTOS RECUPERADOS:\n{contexto}\n\n'
    'PERGUNTA: {pergunta}\n\n'
    'RESPOSTA (cite as fontes):'
)

if USE_VLLM:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model='meta-llama/Llama-3.1-8B-Instruct',
        base_url=VLLM_URL,
        api_key='dummy',
        temperature=0.1,
        max_tokens=1024,
    )
    print('✅ vLLM conectado')
else:
    # LLM simulado para demonstração (sem vLLM)
    from langchain_core.language_models.fake import FakeListLLM
    llm = FakeListLLM(responses=[
        'Com base nos documentos: a prisão preventiva exige fundamentação concreta. '
        '[Fonte: hc_stj.pdf]',
        'Com base nos documentos: o compartilhamento de dados do COAF é constitucional. '
        '[Fonte: lgpd_segpublica.pdf]',
        'Com base nos documentos: imagens de videomonitoramento são dados sensíveis. '
        '[Fonte: lgpd_segpublica.pdf]',
    ])
    print('⚠️ Usando LLM simulado (FakeListLLM) — para vLLM real, ajuste USE_VLLM=True')

print("\n✅ Pipeline RAG configurado")

In [ ]:
def rag_juridico(pergunta, top_k=5):
    # Executa pipeline RAG para perguntas juridicas
    # 1. Recuperar contexto
    chunks_relevantes = buscar_chunks(pergunta, top_k=top_k)
    contexto = '\n\n---\n\n'.join(chunks_relevantes)

    # 2. Montar prompt
    prompt_formatado = PROMPT_RAG.format_messages(
        contexto=contexto,
        pergunta=pergunta
    )

    # 3. Gerar resposta
    resposta = llm.invoke(prompt_formatado)
    texto_resposta = resposta.content if hasattr(resposta, 'content') else str(resposta)

    return {
        'pergunta': pergunta,
        'chunks_recuperados': len(chunks_relevantes),
        'contexto_chars': len(contexto),
        'resposta': texto_resposta
    }

# Testar com perguntas jurídicas
perguntas_teste = [
    'Quais são os requisitos para decretação da prisão preventiva?',
    'O compartilhamento de dados do COAF com a polícia precisa de autorização judicial?',
    'Como a LGPD classifica imagens de câmeras de segurança policial?',
]

print("=" * 65)
print("TESTE DO PIPELINE RAG JURÍDICO")
print("=" * 65)

for pergunta in perguntas_teste:
    resultado = rag_juridico(pergunta)
    print(f"\n❓ PERGUNTA: {resultado['pergunta']}")
    print(f"   Chunks recuperados: {resultado['chunks_recuperados']}")
    print(f"   Contexto: {resultado['contexto_chars']} chars")
    print(f"   RESPOSTA: {resultado['resposta'][:400]}")
    print("-" * 65)

## 📊 Etapa 5: Avaliação da Qualidade da Ingestão

Avaliar se o pipeline de ingestão (Docling → chunks) resulta em
boa recuperação de informações jurídicas relevantes.

**⏱️ Tempo estimado: 10 minutos**

In [ ]:
# Avaliacao usando as perguntas-teste do corpus (dataset da aula)
DATASET_PATH = Path('/sessions/nice-kind-pasteur/mnt/MBA - RAG & CAG Aplicados a Direito e Segurança Pública/aula5/datasets/corpus_docling_aula5.json')

if DATASET_PATH.exists():
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    perguntas_eval = dataset.get('perguntas_teste', [])[:5]
else:
    # Perguntas de avaliação mínimas
    perguntas_eval = [
        {'id': 'Q001', 'pergunta': 'Quais os requisitos para prisão preventiva?',
         'documentos_relevantes': ['hc_stj.pdf']},
        {'id': 'Q002', 'pergunta': 'O descumprimento de medida protetiva é crime mesmo com consentimento?',
         'documentos_relevantes': ['lei_maria_penha.pdf']},
    ]

print(f"📊 Avaliando com {len(perguntas_eval)} perguntas de teste...\n")

resultados_eval = []
for pq in perguntas_eval:
    # Buscar chunks relevantes
    chunks_rec = buscar_chunks(pq['pergunta'], top_k=3)

    # Calcular métricas simples
    docs_esperados = pq.get('documentos_relevantes', [])

    # Verificar se algum chunk relevante foi recuperado (por palavras-chave)
    resposta_esperada = pq.get('resposta_esperada', '').lower()
    palavras_chave = resposta_esperada.split()[:10] if resposta_esperada else []

    contexto_total = ' '.join(chunks_rec).lower()
    palavras_encontradas = sum(1 for p in palavras_chave if p in contexto_total)
    relevancia = palavras_encontradas / len(palavras_chave) if palavras_chave else 0.5

    resultados_eval.append({
        'id': pq['id'],
        'pergunta': pq['pergunta'][:60],
        'chunks_recuperados': len(chunks_rec),
        'relevancia_estimada': relevancia
    })

print(f"{"ID":<8} {"Pergunta":<62} {"Chunks":<8} {"Relevancia"}")
print("-" * 90)
for r in resultados_eval:
    barra = "█" * int(r["relevancia_estimada"] * 10) + "░" * (10 - int(r["relevancia_estimada"] * 10))
    print(f"{r['id']:<8} {r['pergunta']:<62} {r['chunks_recuperados']:<8} {barra} {r['relevancia_estimada']:.0%}")

media_relevancia = sum(r['relevancia_estimada'] for r in resultados_eval) / len(resultados_eval)
print(f"\n📊 Relevância média estimada: {media_relevancia:.0%}")
print("\n💡 Para avaliação rigorosa, use RAGAS (veja Aula 10)")

## ✅ Checkpoint Final — Lab 5 e Aula 5


In [ ]:
print("=" * 65)
print("CHECKPOINT FINAL — LAB 5 / AULA 5")
print("=" * 65)
erros = []

if len(chunks_base) == 0:
    erros.append('❌ Nenhum chunk no corpus')
else:
    print(f"✅ Corpus: {len(chunks_base)} chunks indexados")

if embeddings.shape[1] != 1024:
    erros.append(f'❌ Embeddings com dim errado: {embeddings.shape[1]} (esperado 1024)')
else:
    print(f"✅ Embeddings BGE-M3: {embeddings.shape} (dim=1024)")

# Teste rápido de busca
try:
    teste_busca = buscar_chunks('prisão preventiva', top_k=2)
    assert len(teste_busca) > 0
    print("✅ Busca vetorial funcionando")
except Exception as e:
    erros.append(f'❌ Busca falhou: {e}')

# Pipeline RAG
try:
    teste_rag = rag_juridico('teste')
    assert 'resposta' in teste_rag
    print("✅ Pipeline RAG funcionando")
except Exception as e:
    erros.append(f'❌ RAG falhou: {e}')

if erros:
    for e in erros: print(e)
else:
    print("\n🎉 AULA 5 CONCLUÍDA COM SUCESSO!")
    print("\nResume do que você aprendeu:")
    print("  1. Docling: instalação, configuração e conversão de PDFs")
    print("  2. OCR: detecção automática e processamento de documentos escaneados")
    print("  3. Tabelas: extração, conversão para DataFrame e texto natural")
    print("  4. Pipeline em escala: paralelismo, cache e batch embeddings")
    print("  5. RAG completo: Docling + FAISS/OpenSearch + LangChain + vLLM")

## 🏋️ Exercício Final

**Desafio integrador:** Adicione um documento novo ao corpus e verifique se
o sistema RAG consegue responder perguntas sobre ele.

Passos:
1. Crie um PDF sobre a Súmula 512/STJ (tráfico hediondo)
2. Processe com Docling e gere chunks
3. Adicione ao índice FAISS
4. Faça a pergunta: `'A aplicação do art. 33 §4º afasta a hediondez do tráfico?'`
5. Avalie se a resposta está correta baseada na Súmula 512


## 📖 Referências (ABNT)

AUER, Peter et al. **Docling Technical Report**. arXiv:2408.09869, 2024.

CHEN, J. et al. **BGE M3-Embedding: Multi-Lingual, Multi-Functionality, Multi-Granularity**.
arXiv:2309.07597, 2024.

LEWIS, Patrick et al. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks**.
*NeurIPS*, v. 33, p. 9459-9474, 2020.

LANGCHAIN. **LangChain Python Documentation**. Disponível em: <https://python.langchain.com>.
Acesso em: abr. 2026.
